## 加载数据集

In [6]:
from datasets import load_dataset

#仅加载训练集
ds = load_dataset('squad', split='train')

Using the latest cached version of the module from C:\Users\HP\.cache\huggingface\modules\datasets_modules\datasets\squad\d6ec3ceb99ca480ce37cdd35555d6cb2511d223b9150cce08a837ef62ffea453 (last modified on Fri Aug 25 15:47:36 2023) since it couldn't be found locally at squad., or remotely on the Hugging Face Hub.
Found cached dataset squad (C:/Users/HP/.cache/huggingface/datasets/squad/plain_text/1.0.0/d6ec3ceb99ca480ce37cdd35555d6cb2511d223b9150cce08a837ef62ffea453)


In [4]:
ds = load_dataset('squad')
ds

Found cached dataset squad (C:/Users/HP/.cache/huggingface/datasets/squad/plain_text/1.0.0/d6ec3ceb99ca480ce37cdd35555d6cb2511d223b9150cce08a837ef62ffea453)
100%|████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 42.88it/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [3]:
ds = load_dataset('squad', split='train')
ds

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 87599
})

## 加载开源数据集，并转换格式

In [5]:
from datasets import load_dataset

ALPACA_PROMPT_DICT = {
    "prompt_input": (
        "Below is an instruction that describes a task, paired with an input that provides further context. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response: "
    ),
    "prompt_no_input": (
        "FFF Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{instruction}\n\n### Response: "
    ),
}

def extract_alpaca_dataset(example):
    if example.get("input", "") != "":
        prompt_format = ALPACA_PROMPT_DICT["prompt_input"]
    else:
        prompt_format = ALPACA_PROMPT_DICT["prompt_no_input"]
    return {'input': prompt_format.format(**example)}

def load_data(dataset_name):
    if dataset_name == 'alpaca':
        return load_dataset("tatsu-lab/alpaca")
    elif dataset_name == 'alpaca-clean':
        return load_dataset("yahma/alpaca-cleaned")
    elif dataset_name == 'squad':
        return load_dataset("squad")
    elif dataset_name == 'chip2':
        return load_dataset("laion/OIG", data_files='unified_chip2.jsonl')
    elif dataset_name == 'self-instruct':
        return load_dataset("yizhongw/self_instruct", name='self_instruct')
    elif dataset_name == 'hh-rlhf':
        return load_dataset("Anthropic/hh-rlhf")
    elif dataset_name == 'longform':
        return load_dataset("akoksal/LongForm")
    elif dataset_name == 'oasst1':
        return load_dataset("timdettmers/openassistant-guanaco")
    elif dataset_name == 'vicuna':
        raise NotImplementedError("Vicuna data was not released.")
    else:
        if os.path.exists(dataset_name):
            try:
                args.dataset_format = args.dataset_format if args.dataset_format else "input-output"
                full_dataset = local_dataset(dataset_name)
                return full_dataset
            except:
                raise ValueError(f"Error loading dataset from {dataset_name}")
        else:
            raise NotImplementedError(f"Dataset {dataset_name} not implemented yet.")

def format_dataset(dataset, dataset_format):
    if (dataset_format == 'alpaca' or dataset_format == 'alpaca-clean'):
        dataset = dataset.map(extract_alpaca_dataset, remove_columns=['instruction'])
    elif dataset_format == 'chip2':
        dataset = dataset.map(lambda x: {
            'input': x['text'].split('\n<bot>: ')[0].replace('<human>: ', ''),
            'output': x['text'].split('\n<bot>: ')[1],
        })
    elif dataset_format == 'self-instruct':
        for old, new in [["prompt", "input"], ["completion", "output"]]:
            dataset = dataset.rename_column(old, new)
    elif dataset_format == 'hh-rlhf':
        dataset = dataset.map(lambda x: {
            'input': '',
            'output': x['chosen']
        })
    elif dataset_format == 'oasst1':
        dataset = dataset.map(lambda x: {
            'input': '',
            'output': x['text'],
        })
    elif dataset_format == 'input-output':
        # leave as is
        pass
    # Remove unused columns.
    dataset = dataset.remove_columns(
        [col for col in dataset.column_names['train'] if col not in ['input', 'output']]
    )
    return dataset

### alpaca

In [6]:
dataset_old = load_data('alpaca')
dataset_new = format_dataset(dataset_old, dataset_format='alpaca')

print(dataset_old['train'][0])
print("-"*50)
print(dataset_new['train'][0])

Found cached dataset parquet (C:/Users/HP/.cache/huggingface/datasets/tatsu-lab___parquet/tatsu-lab--alpaca-2b32f0433506ef5f/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)
100%|███████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 170.02it/s]
                                                                                                                       

{'instruction': 'Give three tips for staying healthy.', 'input': '', 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}
--------------------------------------------------
{'input': 'FFF Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response: ', 'output': '1.Eat a balanced diet and make sure to include plenty 

In [4]:
print(dataset_new['train'][0])

{'input': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response: ', 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}


### oasst1

In [13]:
dataset_old = load_data('oasst1')
dataset_new = format_dataset(dataset_old, dataset_format='oasst1')

print(dataset_old['train'][18])
print("-"*50)
print(dataset_new['train'][18])

Found cached dataset json (C:/Users/HP/.cache/huggingface/datasets/timdettmers___json/timdettmers--openassistant-guanaco-c93588435bc90172/0.0.0/e347ab1c932092252e717ff3f949105a4dd28b27e842dd53157d2f72e276c2e4)
100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 276.25it/s]
Loading cached processed dataset at C:\Users\HP\.cache\huggingface\datasets\timdettmers___json\timdettmers--openassistant-guanaco-c93588435bc90172\0.0.0\e347ab1c932092252e717ff3f949105a4dd28b27e842dd53157d2f72e276c2e4\cache-ae8ddaf4cb8b94bf.arrow
Loading cached processed dataset at C:\Users\HP\.cache\huggingface\datasets\timdettmers___json\timdettmers--openassistant-guanaco-c93588435bc90172\0.0.0\e347ab1c932092252e717ff3f949105a4dd28b27e842dd53157d2f72e276c2e4\cache-d42bb5439d4ddf4a.arrow


{'text': "### Human: Write a 4chan style greentext about someone who loves the new romantic comedy movie, with an ironic twist that re-contextualizes the story at the end.. It should start with '>be me'### Assistant: >be me\n>sister wants to watch the new hit romcom\n>not a fan, but what the hell, its got a fantasy theme\n>we're watching, but a sex scene comes up\n>then another\n>i keep looking away from the screen\n>lotsofsex.jpg\n>sister starts teasing me for my discomfort \n\nSo anyway, how did you get into Game of Thrones?"}
--------------------------------------------------
{'input': '', 'output': "### Human: Write a 4chan style greentext about someone who loves the new romantic comedy movie, with an ironic twist that re-contextualizes the story at the end.. It should start with '>be me'### Assistant: >be me\n>sister wants to watch the new hit romcom\n>not a fan, but what the hell, its got a fantasy theme\n>we're watching, but a sex scene comes up\n>then another\n>i keep looking a

## SQuAD数据集访问

In [10]:
from datasets import load_dataset
dataset = load_dataset("squad")
dataset

Found cached dataset squad (C:/Users/HP/.cache/huggingface/datasets/squad/plain_text/1.0.0/d6ec3ceb99ca480ce37cdd35555d6cb2511d223b9150cce08a837ef62ffea453)
100%|████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  6.97it/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [11]:
dataset['train']

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 87599
})

In [12]:
dataset['train'][0]

{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

In [13]:
def map_function(example):
     # Get the question and model output
     question = f"#### Human: {example['question'].strip()}"
     output = f"#### Assistant: {example['answers']['text'][0].strip()}"

     print(question, output)

data_train = dataset["train"].map(map_function)

Map:   1%|▍                                                               | 605/87599 [00:00<00:16, 5282.03 examples/s]

#### Human: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? #### Assistant: Saint Bernadette Soubirous
#### Human: What is in front of the Notre Dame Main Building? #### Assistant: a copper statue of Christ
#### Human: The Basilica of the Sacred heart at Notre Dame is beside to which structure? #### Assistant: the Main Building
#### Human: What is the Grotto at Notre Dame? #### Assistant: a Marian place of prayer and reflection
#### Human: What sits on top of the Main Building at Notre Dame? #### Assistant: a golden statue of the Virgin Mary
#### Human: When did the Scholastic Magazine of Notre dame begin publishing? #### Assistant: September 1876
#### Human: How often is Notre Dame's the Juggler published? #### Assistant: twice
#### Human: What is the daily student paper at Notre Dame called? #### Assistant: The Observer
#### Human: How many student news papers are found at Notre Dame? #### Assistant: three
#### Human: In what year did the student paper Common 

Map:   2%|█▍                                                             | 2020/87599 [00:00<00:15, 5632.22 examples/s]

#### Human: What was the primary concern of other nations who objected to including political groups in the definition of genocide? #### Assistant: international intervention in domestic politics
#### Human: Which distinguished academic of genocide highlighted several other countries opposed to including political genocide in the Convention? #### Assistant: William Schabas
#### Human: What atrocity motivated a self-serving USSR to object to the provision of political killings drafted into the Convention? #### Assistant: its own Great Purge
#### Human: Which group was accused by the ECHR of having an overly constricted idea of the meaning of destruction in defining genocide? #### Assistant: majority of legal scholars
#### Human: What groups did the ECHR feel should be included as potential victims of genocide? #### Assistant: national, racial, religious or ethnic
#### Human: In 2007 what former case did the European Court of Human Rights draw on to further refine qualifiers of genocide?

Map:   4%|██▎                                                            | 3264/87599 [00:00<00:14, 5958.39 examples/s]

#### Human: Who imposed on the military might on Tibet in the past? #### Assistant: the Mongols
#### Human: Hok-Lam Chan states that Deshin Skekpa was only invited for what purpose? #### Assistant: religious purposes
#### Human: What was said to be the  Yongle Emperor plans? #### Assistant: to send a military force into Tibet
#### Human: Why was the Yongle Emperor said to have been planning to send military forces into Tibet? #### Assistant: to forcibly give the Karmapa authority over all the Tibetan Buddhist schools
#### Human: Who convinced the Yongle emperor not to send military forces into Tibet? #### Assistant: Deshin Shekpa
#### Human: Who did the Yongle Emperor have a conflict with? #### Assistant: Jianwen Emperor
#### Human: Who aided the legitimacy of the Yongle Emperor's rule? #### Assistant: Deshin Shekpa
#### Human: What served as a legitimizing tool for the emperor's succession to the throne? #### Assistant: Deshin Shekpa's miracles
#### Human: According to Norbu who faile

... (129 outputs left out)

#### Human: What was Issac Backus' profession? #### Assistant: Baptist minister
#### Human: What region did Isaac Backus come from? #### Assistant: New England
#### Human: When did Isaac Backus write against a state sanctioned religion? #### Assistant: 1773
#### Human: When was Thomas Jefferson's Virginia Statute for Religious Freedom enacted? #### Assistant: 1786
#### Human: Who was Roger Williams? #### Assistant: Baptist theologian
#### Human: What phrase did Roger Williams first use? #### Assistant: "[A] hedge or wall of separation between the garden of the church and the wilderness of the world"
#### Human: What colony was Roger Williams the founder of? #### Assistant: Rhode Island
#### Human: When was the book The Bloody Tenent of Persecution published? #### Assistant: 1644
#### Human: Who used William's phrase as a description of the First Amendment and its restriction on the legislative branch? #### Assistant: Thomas Jefferson
#### Human: Whose conceptions of separation have lon

Map:  99%|█████████████████████████████████████████████████████████████▍| 86759/87599 [00:14<00:00, 6723.15 examples/s]

#### Human: Who was Chief of Staff of the Army before Eisenhower? #### Assistant: Marshall
#### Human: About how many soldiers demobilized after the war ended? #### Assistant: millions
#### Human: What president ignored Eisenhower's recommendations in regard to atomic weapons? #### Assistant: Truman
#### Human: What was Eisenhower's position on the use of nuclear weapons against Japan? #### Assistant: opposed
#### Human: What city did Eisenhower notably visit in 1945? #### Assistant: Warsaw
#### Human: When was it suggested to Eisenhower that he might one day become president? #### Assistant: June 1943
#### Human: Along with dogcatcher, what political job did Eisenhower specifically not want to be considered for? #### Assistant: Grand High Supreme King of the Universe
#### Human: What role did Eisenhower believe he could not fulfill if he was believed to want to become involved in politics? #### Assistant: Army Chief of Staff
#### Human: What general was considered a potential Republic

#### Human: What were the the only land mammal in New Zealand? #### Assistant: bats
#### Human: What is New Zealand made up of? #### Assistant: islands
#### Human: Game animals were introduced here by whom? #### Assistant: acclimatisation societies
#### Human: What resulted having no natural predators for the animals introduced? #### Assistant: their population exploded
#### Human: What was hunting regarded as in British India? #### Assistant: regal sport
#### Human: What does the Indian word "shikaris" mean in English? #### Assistant: big-game hunters
#### Human: What did British officers maintain whole corps of? #### Assistant: shikaris
#### Human: Why did the regals recruit low-ranking local tribes when hunting? #### Assistant: because of their traditional knowledge of the environment and hunting techniques
#### Human: What could a Bengal tiger be hunted from the back of? #### Assistant: an elephant
#### Human: Where was hunting reguarded as a regal sport? #### Assistant: British In